# Dataset merging

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import make_valid

from andeangc import config as cfg

## Sub-regional institutional raw data

In [ ]:
# Define datasets to load
datasets = {
  'camels_cl': (cfg.RESOURCES / 'CAMELS_CL/CAMELS_CL_metadata.csv',
                cfg.RESOURCES / 'CAMELS_CL/CAMELS_CL_daily_1950_2025.csv',
                cfg.RESOURCES / 'CAMELS_CL/basins_CAMELS_CL.gpkg'),
  'pmetobs':   (cfg.RESOURCES / 'PMET_OBS/Q_PMETobs_v11_metadata.csv', 
                cfg.RESOURCES / 'PMET_OBS/Q_PMETobs_1950_2025_v11d.csv', 
                cfg.RESOURCES / 'PMET_OBS/basins_PMETobs_v11.gpkg'),
  'snhi_arg':  (cfg.RESOURCES / 'SNHI_ARG/SNHI_metadata.csv', 
                cfg.RESOURCES / 'SNHI_ARG/SNHI_data.csv', 
                cfg.RESOURCES / 'SNHI_ARG/basins_SNHI.gpkg'),
  'senamhi':   (cfg.RESOURCES / 'SENAMHI_PERU/SENAMHI_metadata.csv', 
                cfg.RESOURCES / 'SENAMHI_PERU/SENAMHI_data.csv', 
                cfg.RESOURCES / 'SENAMHI_PERU/basins_SENAMHI.gpkg')}

# Load all datasets, keyed by source name: metadata["camels_cl"], data["snhi_arg"], ...
metadata, data, shape = {}, {}, {}

for name, (meta_file, data_file, shape_file) in datasets.items():
    
    metadata[name] = pd.read_csv(meta_file, index_col=0)
    data[name] = pd.read_csv(data_file, index_col=0, parse_dates=["date"]).loc[cfg.period_q[0]:cfg.period_q[1]]
    shape[name] = gpd.read_file(shape_file)[["gauge_id", "geometry"]].set_index("gauge_id")


## Preprocessing / homogenize format

In [ ]:
# Standardize gauge_id format. Only CAMELS-CL arrives with bare DGA station codes;
# the other three are prefixed upstream in nb01.
metadata["camels_cl"].index = ['X' + str(i).zfill(8) for i in metadata["camels_cl"].index]
shape["camels_cl"].index    = ['X' + str(i).zfill(8) for i in shape["camels_cl"].index]
data["camels_cl"].columns   = ['X' + str(i).zfill(8) for i in data["camels_cl"].columns]

# Add gauge_name for SNHI (needs original columns before drop)
metadata["snhi_arg"]["gauge_name"] = metadata["snhi_arg"].river + " " + metadata["snhi_arg"].place

# Keep only essential base columns first (avoids fragmentation on wide DataFrames)
base_attrs = ["gauge_lat", "gauge_lon", "gauge_name"]
for meta in metadata.values():
    meta.drop(columns=meta.columns.difference(base_attrs), inplace=True)

# Add institution/country/dataset to now-slim DataFrames
metadata["snhi_arg"][["institution",  "country", "dataset"]]  = ["SNHI",    "Argentina", "Andean-GC (self-produced)"]
metadata["camels_cl"][["institution", "country", "dataset"]]  = ["DGA",     "Chile",     "CAMELS-cl (updated)"]
metadata["senamhi"][["institution",   "country", "dataset"]]  = ["SENAMHI", "Peru",      "Andean-GC (self-produced)"]
metadata["pmetobs"][["institution",   "country", "dataset"]]  = ["PMETobs", "Multiple",  "PMETobs v11 (updated)"]

# Fix invalid geometries
shape["camels_cl"]["geometry"] = make_valid(shape["camels_cl"].geometry)


## Concatenation and saving

In [ ]:
# concatenate. The order is load-bearing: drop_duplicates(keep="last") means a gauge
# present in two sources is taken from the later one — PMET-obs wins over CAMELS-CL.
merge_order = ["snhi_arg", "camels_cl", "pmetobs", "senamhi"]

AndeanGC_metadata = pd.concat([metadata[name] for name in merge_order]).reset_index()
AndeanGC_metadata = AndeanGC_metadata.drop_duplicates(subset=["index"], keep="last")
AndeanGC_metadata = AndeanGC_metadata.rename(columns={"index": "gauge_id"}).set_index("gauge_id")
AndeanGC_metadata["gauge_name"] = AndeanGC_metadata["gauge_name"].str.replace("_", " ").str.title()

# Filter conditions for each dataset (gives priority to PMET)
arg_filter   = (AndeanGC_metadata.dataset == "Andean-GC (self-produced)") & (AndeanGC_metadata.country == "Argentina")
peru_filter  = (AndeanGC_metadata.dataset == "Andean-GC (self-produced)") & (AndeanGC_metadata.country == "Peru")
chile_filter = AndeanGC_metadata.dataset == "CAMELS-cl (updated)"

AndeanGC_data = pd.concat([
    data["snhi_arg"][AndeanGC_metadata[arg_filter].index],
    data["senamhi"][AndeanGC_metadata[peru_filter].index],
    data["camels_cl"][AndeanGC_metadata[chile_filter].index],
    data["pmetobs"]], axis=1)

AndeanGC_shape = pd.concat([
    shape["snhi_arg"].loc[AndeanGC_metadata[arg_filter].index],
    shape["senamhi"].loc[AndeanGC_metadata[peru_filter].index],
    shape["camels_cl"].loc[AndeanGC_metadata[chile_filter].index],
    shape["pmetobs"]])
AndeanGC_shape = pd.concat([AndeanGC_metadata, AndeanGC_shape], axis=1)
AndeanGC_shape = gpd.GeoDataFrame(AndeanGC_shape, geometry='geometry')


In [ ]:
# Save processed data
AndeanGC_metadata.to_csv(cfg.VERSION / 'AndeanGC_metadata.csv')
AndeanGC_data.to_csv(cfg.VERSION / 'AndeanGC_data.csv')
AndeanGC_shape.to_file(cfg.VERSION / 'AndeanGC_shape.gpkg')